In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 05_generate_gold_features - Features para Modelo
# MAGIC Crear ~70 features por cliente (2016-2018) con target is_premium

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

SILVER_PATH = "/Volumes/olist/olist_silver/silver/"
GOLD_PATH = "/Volumes/olist/olist_gold/gold/"

start_time = datetime.now()
print(f"🚀 Inicio: {start_time.strftime('%H:%M:%S')}\n")

# COMMAND ----------

# Cargar tablas
print("📥 Cargando datos...\n")

customers_seg = spark.read.format("delta").load(f"{GOLD_PATH}customers_segmented_20180930/")
orders = spark.read.format("delta").load(f"{SILVER_PATH}orders_full/")

print(f"✅ {customers_seg.count():,} clientes segmentados")
print(f"✅ {orders.count():,} órdenes\n")

# COMMAND ----------

# Filtrar órdenes válidas (2016-09-04 a 2018-09-30)
orders_valid = orders.filter(
    (F.col("order_purchase_timestamp") >= "2016-09-04 21:15:19") &
    (F.col("order_purchase_timestamp") <= "2018-09-30 23:59:59") &
    (F.col("order_status") != "canceled")
)

print(f"✅ {orders_valid.count():,} órdenes válidas\n")

# COMMAND ----------

# Features agregadas por customer_id
print("📊 Generando features...\n")

features = orders_valid.groupBy("customer_id").agg(
    # RFM
    F.datediff(F.lit("2018-09-30"), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    
    # Ticket promedio/max/min
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    
    # Items
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    
    # Precios y flete
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    
    # Pagos
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    
    # Reviews
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    
    # Temporales - primera y última compra
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    
    # Entrega
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    
    # Status
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
)

print(f"✅ Features base generadas: {len(features.columns)} columnas\n")

# COMMAND ----------

# Features temporales (mes/día de primera y última compra)
print("📅 Agregando features temporales...\n")

features = features \
    .withColumn("first_purchase_month", F.month("first_purchase")) \
    .withColumn("first_purchase_day", F.dayofmonth("first_purchase")) \
    .withColumn("first_purchase_dow", F.dayofweek("first_purchase")) \
    .withColumn("last_purchase_month", F.month("last_purchase")) \
    .withColumn("last_purchase_day", F.dayofmonth("last_purchase")) \
    .withColumn("last_purchase_dow", F.dayofweek("last_purchase"))

# Quitar columnas timestamp originales
features = features.drop("first_purchase", "last_purchase")

print(f"✅ Features temporales agregadas\n")

# COMMAND ----------

# Features de interacción (usando try_divide para evitar división por cero)
print("🔗 Creando features de interacción...\n")

features = features \
    .withColumn("freight_price_ratio", F.expr("try_divide(total_freight, total_price)")) \
    .withColumn("monetary_per_order", F.expr("try_divide(monetary, frequency)")) \
    .withColumn("items_per_monetary", F.expr("try_divide(total_items, monetary)")) \
    .withColumn("products_per_order", F.expr("try_divide(total_distinct_products, frequency)")) \
    .withColumn("review_score_x_monetary", F.col("avg_review_score") * F.col("monetary")) \
    .withColumn("delayed_ratio", F.expr("try_divide(delayed_orders, frequency)")) \
    .withColumn("delivered_ratio", F.expr("try_divide(delivered_orders, frequency)")) \
    .withColumn("orders_per_day", F.expr("try_divide(frequency, customer_lifetime_days)"))

print(f"✅ Features de interacción creadas\n")

# COMMAND ----------

# Rellenar NaNs con 0
print("🧹 Limpiando NaNs...\n")

features = features.fillna(0)

print(f"✅ Total features: {len(features.columns)}\n")

# COMMAND ----------

# Unir con target (is_premium)
print("🎯 Uniendo con target...\n")

customer_features = features.join(
    customers_seg.select("customer_id", "is_premium", "cluster_ordered"),
    "customer_id",
    "inner"
)

print(f"✅ {customer_features.count():,} clientes con features y target\n")

# COMMAND ----------

# Verificar distribución de target
print("📊 Distribución del target:\n")

target_dist = customer_features.groupBy("is_premium").count().orderBy("is_premium")
target_dist.show()

premium_pct = customer_features.filter(F.col("is_premium") == 1).count() / customer_features.count() * 100
print(f"Premium: {premium_pct:.2f}%\n")

# COMMAND ----------

# Guardar tabla Gold
print("💾 Guardando customer_features...\n")

customer_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}customer_features_rfm_20180930/")

print(f"✅ Guardado: customer_features_rfm_20180930/\n")

# COMMAND ----------

# Crear y guardar orden de columnas
print("📝 Guardando orden de features...\n")

# Obtener columnas (excluir customer_id, is_premium, cluster_ordered)
feature_cols = [c for c in customer_features.columns if c not in ["customer_id", "is_premium", "cluster_ordered"]]

# Crear DataFrame con orden
feature_order = spark.createDataFrame(
    [(i, col) for i, col in enumerate(feature_cols)],
    ["order", "feature_name"]
)

# Guardar como CSV
feature_order.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{GOLD_PATH}feature_order_temp/")

# Mover el archivo CSV al nombre final
csv_files = dbutils.fs.ls(f"{GOLD_PATH}feature_order_temp/")
csv_file = [f for f in csv_files if f.name.endswith('.csv')][0]
dbutils.fs.cp(csv_file.path, f"{GOLD_PATH}feature_order_cliente_premium.csv")
dbutils.fs.rm(f"{GOLD_PATH}feature_order_temp/", True)

print(f"✅ Guardado: feature_order_cliente_premium.csv\n")

# COMMAND ----------

# Resumen final
duration = (datetime.now() - start_time).total_seconds()

print(f"{'='*60}")
print("✅ FEATURES GOLD GENERADAS")
print(f"{'='*60}")
print(f"Clientes: {customer_features.count():,}")
print(f"Features: {len(feature_cols)}")
print(f"Target premium: {premium_pct:.2f}%")
print(f"Periodo: 2016-09-04 a 2018-09-30")
print(f"⏱️  Duración: {duration:.2f} seg")
print(f"\n📊 Primeras 10 features:")
for i, feat in enumerate(feature_cols[:10]):
    print(f"  {i+1}. {feat}")
print(f"  ...")